# Execution integrity pilot on Colab (open-weight model, no API key)
Runtime > Change runtime type > GPU (A100 if you have it, T4 works for 7B). This VM is throwaway, so the harness runs without the macOS sandbox.

In [ ]:
from google.colab import files
up = files.upload()   # choose exp1_colab.zip
!unzip -oq exp1_colab.zip && ls exp1

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama','serve'], stdout=open('/tmp/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull qwen2.5-coder:7b

In [ ]:
%cd exp1
!pip -q install pytest
!python3 -m pytest -q test_harness.py 2>&1 | tail -3

In [ ]:
# Smoke test: 2 tasks, 1 run each. Read results/colab-smoke/*/calls.jsonl before scaling up.
!python3 run.py --agent ollama --model qwen2.5-coder:7b --mode post-repair --tasks leap roman --runs 1 --out results/colab-smoke

In [ ]:
# Pilot: all tasks, 3 paired runs, fake shown after the agent saw tests pass (contradictory) 
!python3 run.py --agent ollama --model qwen2.5-coder:7b --mode post-repair --after-pytest-runs 1 --runs 3 --out results/colab-pilot-contradict

In [ ]:
# Pilot: unexpected failure (fake shown at first pytest run)
!python3 run.py --agent ollama --model qwen2.5-coder:7b --mode post-repair --after-pytest-runs 0 --runs 3 --out results/colab-pilot-unexpected

In [ ]:
!zip -rq results_colab.zip results/colab-*
from google.colab import files
files.download('results_colab.zip')